In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine

import pandas as pd, numpy as np
BASE  = '/content/drive/MyDrive/Palestine_Project'
RAW   = f'{BASE}/00_raw/Palestine_Project_Data.xlsx'
CLEAN = f'{BASE}/01_clean'
ML    = f'{BASE}/02_ml'

def read_sheet(name, **kw):
    return pd.read_excel(RAW, sheet_name=name, engine='calamine', **kw)

def save_clean(df, filename):
    df.to_csv(f'{CLEAN}/{filename}', index=False, encoding='utf-8')
    print(f'saved {filename}: {len(df):,} rows x {df.shape[1]} cols')

In [ ]:
df = read_sheet('2_Damage_Sites_Gaza')
assert len(df) == 192851

df.insert(0, 'site_key', np.arange(1, len(df) + 1, dtype='int32'))
assert df[['lon', 'lat']].duplicated().sum() == 0

df = df.rename(columns={'SensorDate': 'SensorDate_1', 'SensorID': 'SensorID_1',
                        'ConfidenceID': 'ConfidenceID_1', 'Main_Damage_Site_Class': 'Main_Damage_Site_Class_1'})
df['Damage_Status_1'] = np.nan

parts = []
for r in range(1, 14):
    p = pd.DataFrame({
        'site_key':      df['site_key'],
        'round_no':      np.int8(r),
        'sensor_date':   pd.to_datetime(df[f'SensorDate_{r}'], utc=True, errors='coerce').dt.date,
        'sensor_id':     df[f'SensorID_{r}'],
        'confidence_id': df[f'ConfidenceID_{r}'],
        'damage_class':  df[f'Main_Damage_Site_Class_{r}'],
        'damage_status': df[f'Damage_Status_{r}'],
    })
    parts.append(p[p['damage_class'].notna()])
fact = pd.concat(parts, ignore_index=True)
for c in ['sensor_id', 'confidence_id', 'damage_class', 'damage_status']:
    fact[c] = fact[c].astype('Int16')

gov = {'North Gaza': 'PS0255', 'Gaza': 'PS0260', 'Deir Al-Balah': 'PS0265', 'Khan Yunis': 'PS0270', 'Rafah': 'PS0275'}
assert set(df['Governorate'].dropna().unique()) <= set(gov), set(df['Governorate'].unique()) - set(gov)
C = df[[f'Main_Damage_Site_Class_{r}' for r in range(1, 14)]]
prev = C.shift(axis=1).ffill(axis=1)
site = pd.DataFrame({
    'site_key': df['site_key'], 'source_site_group_id': df['SiteID'],
    'lon': df['lon'], 'lat': df['lat'],
    'geo_pcode': df['Governorate'].map(gov), 'municipality': df['Municipality'], 'neighborhood': df['Neighborhood'],
    'site_note': df['Notes'], 'event_code_raw': df['EventCode'],
})
site['event_code_std']          = 'CE20231007PSE'
site['event_code_suspect_flag'] = (df['EventCode'] == 'CE20251007PSE').astype(int)
site['first_round_no']          = C.notna().values.argmax(axis=1) + 1
site['final_class']             = df['Main_Damage_Site_Class_13'].astype('Int16')
site['has_class_improvement_flag'] = (C > prev).any(axis=1).astype(int)

rnd = fact.groupby('round_no').agg(sensor_date_min=('sensor_date', 'min'), sensor_date_max=('sensor_date', 'max'),
                                   n_sites_assessed=('site_key', 'size')).reset_index()
cls = pd.DataFrame({'damage_class': [1, 2, 3, 4, 6, 11],
                    'class_label': ['Destroyed', 'Severely damaged', 'Moderately damaged', 'Possibly damaged',
                                    'Undocumented class 6 (verify in UNOSAT docs)', 'Undocumented class 11 (verify in UNOSAT docs)'],
                    'severity_rank': [1, 2, 3, 4, 5, 5]})

print('sites', len(site), '| long rows', len(fact), '(expected 1,648,583)')
print('rows per round:', rnd['n_sites_assessed'].tolist())
print('round 13 classes:', fact[fact['round_no'] == 13]['damage_class'].value_counts().sort_index().to_dict())
print('sites first seen in round 13 only:', int((site['first_round_no'] == 13).sum()),
      '| suspect event codes:', int(site['event_code_suspect_flag'].sum()),
      '| class improvements:', int(site['has_class_improvement_flag'].sum()),
      '| no municipality:', int(site['municipality'].isna().sum()))
save_clean(site, 'dim_damage_site.csv'); save_clean(rnd, 'dim_damage_round.csv')
save_clean(cls, 'dim_damage_class.csv'); save_clean(fact, 'fact_damage_site_round.csv')